# COVID-19 Data Trend Analysis

**Subject:** Data Mining & Data Warehouse  
**Dataset:** Johns Hopkins CSSE COVID-19 Time-Series  
**Novel Technique:** Epidemic Trajectory Fingerprinting (ETF)

---

## 1. Introduction

COVID-19, caused by the SARS-CoV-2 virus, emerged in late 2019 and rapidly escalated into a global pandemic. Understanding the patterns of its spread — peaks, plateaus, waves, and regional differences — is crucial for public health decision-making.

This project applies **data mining techniques** to COVID-19 time-series data to:
- Identify trends in confirmed cases, deaths, and recoveries.
- Compare pandemic trajectories across countries and regions.
- Discover hidden patterns using clustering and association mining.
- Propose a **novel hybrid technique** — *Epidemic Trajectory Fingerprinting (ETF)* — that combines wavelet decomposition, Dynamic Time Warping, spectral embedding, and density-based clustering.

### Objectives
1. Preprocess and clean the raw JHU CSSE dataset.
2. Perform exploratory data analysis (EDA).
3. Apply standard data mining techniques: rolling averages, STL decomposition, correlation analysis, K-Means clustering, and Apriori pattern mining.
4. Develop and demonstrate the novel ETF method.
5. Derive actionable insights about peak periods, most-affected regions, and the impact of interventions.

## 2. Dataset Description

**Source:** [Johns Hopkins CSSE COVID-19 Data Repository](https://github.com/CSSEGISandData/COVID-19)  
**Format:** Three CSV files (confirmed, deaths, recovered) in wide format  
**Coverage:** 200+ countries/regions  
**Time Range:** January 22, 2020 — March 9, 2023 (repository archived)  

| File | Description |
|------|-------------|
| `time_series_covid19_confirmed_global.csv` | Cumulative confirmed cases per country per day |
| `time_series_covid19_deaths_global.csv` | Cumulative deaths per country per day |
| `time_series_covid19_recovered_global.csv` | Cumulative recovered cases per country per day |

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────────
import sys, os, warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Project modules
from src import data_ingest, eda, standard_techniques, etf, visualization

# Plotly notebook mode
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✔ All imports successful')

## 3. Data Preprocessing

In [ ]:
# ── 3.1 Load & Merge Data ────────────────────────────────────────────────
df = data_ingest.load_merged()
print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')
print(f'Countries: {df["country"].nunique()}')
df.head(10)

In [ ]:
# ── 3.2 Data Info & Missing Values ───────────────────────────────────────
print('Column types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nBasic statistics:')
df.describe()

In [ ]:
# ── 3.3 Select Focus Countries ──────────────────────────────────────────
TOP_N = 15
top_countries = data_ingest.get_top_countries(df, metric='confirmed', n=TOP_N)
print(f'Top {TOP_N} countries by confirmed cases:')
for i, c in enumerate(top_countries, 1):
    print(f'  {i:2d}. {c}')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── 4.1 Top Countries Table ──────────────────────────────────────────────
top_table = eda.top_countries_table(df, n=20)
top_table

In [ ]:
# ── 4.2 Summary Statistics for Top Countries ─────────────────────────────
stats = eda.summary_statistics(df, metric='confirmed')
stats.head(20)

In [ ]:
# ── 4.3 Top Countries Bar Chart ─────────────────────────────────────────
fig = visualization.plot_top_countries_bar(df, metric='confirmed', n=20)
fig.show()

In [ ]:
# ── 4.4 Cumulative Confirmed Cases — Top 10 ─────────────────────────────
fig = visualization.plot_cumulative(df, top_countries[:10], metric='confirmed')
fig.show()

In [ ]:
# ── 4.5 Cumulative Deaths — Top 10 ──────────────────────────────────────
fig = visualization.plot_cumulative(df, top_countries[:10], metric='deaths',
                                    title='Cumulative Deaths')
fig.show()

In [ ]:
# ── 4.6 World Choropleth Map ─────────────────────────────────────────────
fig = visualization.plot_choropleth(df, metric='confirmed')
fig.show()

In [ ]:
# ── 4.7 Growth Rate — India ──────────────────────────────────────────────
growth = eda.compute_growth_rate(df, 'India')
growth_smooth = growth.rolling(7).mean()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(growth_smooth, linewidth=1)
ax.set_title('Daily Growth Rate (7-day avg) — India')
ax.set_ylabel('Growth Rate')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. Techniques Used

### 5.1 Time Series Analysis — Rolling Averages

In [ ]:
# ── 5.1 Daily Cases with Rolling Average ─────────────────────────────────
fig = visualization.plot_daily(df, ['US', 'India', 'Brazil'],
                               rolling_window=7)
fig.show()

In [ ]:
# Rolling average table for US
ra = standard_techniques.rolling_average(df, 'US', windows=[7, 14, 30])
ra.tail(10)

### 5.2 STL Decomposition

STL (Seasonal-Trend decomposition using LOESS) separates a time series into three components:
- **Trend:** Long-term direction
- **Seasonal:** Repeating weekly pattern
- **Residual:** Random noise / anomalies

In [ ]:
# STL Decomposition — US
stl_result = standard_techniques.stl_decompose(df, 'US', period=7)
fig = visualization.plot_stl(stl_result, 'US')
plt.show()

In [ ]:
# STL Decomposition — India
stl_india = standard_techniques.stl_decompose(df, 'India', period=7)
fig = visualization.plot_stl(stl_india, 'India')
plt.show()

### 5.3 Correlation Analysis

We compute Pearson correlation between countries' daily-case curves to find:
- Which countries had similar pandemic trajectories?
- Is there a time lag between waves in different countries?

In [ ]:
# ── Correlation Matrix ───────────────────────────────────────────────────
corr = standard_techniques.correlation_matrix(df, top_countries[:10])
fig = visualization.plot_correlation_heatmap(corr)
plt.show()

In [ ]:
# ── Lagged Cross-Correlation: US vs India ────────────────────────────────
lag_corr = standard_techniques.lagged_cross_correlation(df, 'US', 'India', max_lag=30)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lag_corr.index, lag_corr.values, width=0.8, alpha=0.8)
ax.set_xlabel('Lag (days)')
ax.set_ylabel('Pearson r')
ax.set_title('Lagged Cross-Correlation: US vs India')
ax.axhline(0, color='gray', linestyle='--')
best_lag = lag_corr.idxmax()
ax.axvline(best_lag, color='red', linestyle=':', label=f'Best lag = {best_lag}')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Peak correlation at lag = {best_lag} days (r = {lag_corr[best_lag]:.3f})')

### 5.4 K-Means Clustering

We cluster countries by their pandemic profile features:
- Total confirmed, total deaths
- Peak daily cases, days to peak
- Fatality rate, recovery rate

In [ ]:
# ── Build Features ───────────────────────────────────────────────────────
features = standard_techniques.build_country_features(df)
print(f'Feature matrix: {features.shape}')
features.head(10)

In [ ]:
# ── Elbow Method ─────────────────────────────────────────────────────────
elbow = standard_techniques.elbow_search(features)
fig = visualization.plot_elbow(elbow)
plt.show()
elbow

In [ ]:
# ── K-Means with Optimal K ───────────────────────────────────────────────
BEST_K = elbow.loc[elbow['silhouette'].idxmax(), 'k']
print(f'Best K by silhouette score: {BEST_K}')

labels, pca_2d, sil = standard_techniques.kmeans_cluster(features, k=int(BEST_K))
print(f'Silhouette score: {sil:.3f}')

fig = visualization.plot_kmeans_clusters(pca_2d, labels, features.index.tolist())
plt.show()

In [ ]:
# ── Cluster Membership ───────────────────────────────────────────────────
cluster_df = features.copy()
cluster_df['cluster'] = labels
for cl in sorted(cluster_df['cluster'].unique()):
    members = cluster_df[cluster_df['cluster'] == cl].index.tolist()
    print(f'\nCluster {cl} ({len(members)} countries): {members[:10]}...' if len(members) > 10 else f'\nCluster {cl} ({len(members)} countries): {members}')

### 5.5 Apriori Association Mining

We discretize daily case counts into bands (Low/Medium/High/Surge) and mine frequent patterns:
e.g., *"When the US is in 'Surge', Brazil is also in 'High' with 60% confidence."*

In [ ]:
# ── Apriori ──────────────────────────────────────────────────────────────
focus = ['US', 'India', 'Brazil', 'France', 'Germany', 'United Kingdom']
onehot = standard_techniques.discretize_cases(df, focus)
rules = standard_techniques.run_apriori(onehot, min_support=0.03, min_confidence=0.4)
print(f'Association rules found: {len(rules)}')
if not rules.empty:
    display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))
else:
    print('No rules found — try lowering min_support or min_confidence.')

## 6. Visualization & Results

*(Interactive plots were shown above. Here we add a few more summary visuals.)*

In [ ]:
# ── Deaths Choropleth ────────────────────────────────────────────────────
fig = visualization.plot_choropleth(df, metric='deaths', title='Total Deaths by Country')
fig.show()

In [ ]:
# ── Daily Deaths Smoothed — Top 5 ────────────────────────────────────────
fig = visualization.plot_daily(df, top_countries[:5], metric='daily_deaths',
                               rolling_window=14,
                               title='Daily Deaths (14-day avg) — Top 5 Countries')
fig.show()

## 7. Novel Technique — Epidemic Trajectory Fingerprinting (ETF)

### Motivation

Standard K-Means clustering uses **scalar features** (total cases, peak value, fatality rate) and **loses the shape of the epidemic curve**. Two countries can have the same total cases but vastly different trajectories — one sharp spike vs. a long plateau.

### What makes ETF novel?

ETF is a **hybrid pipeline** that combines four techniques in a way that no existing standard method does:

| Step | Technique | Purpose |
|------|-----------|---------|
| 1 | **Wavelet Decomposition** (DWT, Daubechies-4) | Multi-scale feature extraction |
| 2 | **Dynamic Time Warping** (DTW) | Time-shift invariant distance |
| 3 | **Spectral Embedding** (Laplacian Eigenmaps) | Non-linear dimensionality reduction |
| 4 | **HDBSCAN** | Density-based clustering (no need to choose K) |

### Why not use existing methods?

| Method | Limitation |
|--------|-----------|
| K-Means on scalars | Loses curve shape |
| DTW + K-Means | Single scale only; must choose K |
| Wavelet + K-Means | Not time-shift invariant |
| **ETF (ours)** | **Combines all advantages; no K needed** |

In [ ]:
# ── 7.1 Run the ETF Pipeline ─────────────────────────────────────────────
# Use top 30 countries for a meaningful clustering
etf_countries = data_ingest.get_top_countries(df, metric='confirmed', n=30)
print(f'Running ETF on {len(etf_countries)} countries...')

etf_result = etf.run_etf(
    df, etf_countries,
    metric='daily_confirmed',
    wavelet='db4', level=3,
    n_components=2,
    min_cluster_size=3
)

print(f'Clusters found: {len(set(etf_result["labels"])) - (1 if -1 in etf_result["labels"] else 0)}')
print(f'Noise points: {sum(etf_result["labels"] == -1)}')

In [ ]:
# ── 7.2 DTW Distance Heatmap ─────────────────────────────────────────────
fig = visualization.plot_dtw_heatmap(etf_result['dtw_matrix'], etf_result['countries'])
plt.show()

In [ ]:
# ── 7.3 Spectral Embedding + HDBSCAN Clusters ────────────────────────────
fig = visualization.plot_etf_clusters(etf_result['results_df'])
plt.show()

In [ ]:
# ── 7.4 Trajectory Shapes by Cluster ─────────────────────────────────────
fig = visualization.plot_cluster_curves(etf_result['curves'], etf_result['results_df'])
if fig:
    plt.show()

In [ ]:
# ── 7.5 ETF Cluster Membership ───────────────────────────────────────────
etf_df = etf_result['results_df'].sort_values('cluster')
for cl in sorted(etf_df['cluster'].unique()):
    label = f'Cluster {cl}' if cl >= 0 else 'Noise'
    members = etf_df[etf_df['cluster'] == cl]['country'].tolist()
    print(f'\n{label}: {members}')

### 7.6 ETF vs K-Means — Comparison

Here we compare the clusters found by standard K-Means (scalar features) vs. our novel ETF method (full trajectory shape). The key insight is that ETF groups countries by **how their waves looked**, not just **how many total cases they had**.

In [ ]:
# Compare K-Means vs ETF for the same countries
km_features = standard_techniques.build_country_features(
    df[df['country'].isin(etf_countries)])
km_labels, km_pca, km_sil = standard_techniques.kmeans_cluster(km_features, k=int(BEST_K))

comparison = pd.DataFrame({
    'country': km_features.index,
    'kmeans_cluster': km_labels,
})
comparison = comparison.merge(etf_df[['country', 'cluster']],
                              on='country', how='left')
comparison.rename(columns={'cluster': 'etf_cluster'}, inplace=True)
comparison.sort_values('etf_cluster')

## 8. Insights & Conclusions

### Peak Periods
- The pandemic saw multiple global waves, with major peaks varying by country.
- The US experienced significant surges in winter 2020-21 and during the Omicron wave (Jan 2022).
- India's devastating Delta wave peaked in May 2021.

### Most Affected Regions
- By total cases: US, India, France, Germany, Brazil.
- By fatality rate: Some smaller countries with older demographics showed higher rates.

### Impact of Interventions
- Countries with early lockdowns often appear in different ETF clusters than those with delayed responses.
- The lagged cross-correlation analysis reveals how waves propagated across borders with measurable delays.

### Value of ETF (Novel Technique)
- ETF revealed trajectory families that K-Means missed: countries with similar total counts but different wave patterns were correctly separated.
- The method discovered natural groupings like "Sharp Spike" countries, "Multi-Wave" countries, and "Plateau" countries.
- No need to choose K — HDBSCAN discovers the natural number of trajectory families.

### Future Work
- Incorporate vaccination data as additional features.
- Apply ETF to sub-national (state/province) level data.
- Extend with predictive modelling (ARIMA, Prophet) for forecasting.

In [ ]:
print('═' * 60)
print('  COVID-19 Data Trend Analysis — Complete')
print('═' * 60)